## **Cambois en el Repositorio + git pull de la nueva version**

Si en un contendor en el servidor, se esta desplegando un repositorio que se clono desde GitHub

**¿Que pasaria si hago cambios en el repositorio/proyecto?**

El repositorio siempre modificamos desde develop, o desde una rama especifica (feature/***)

Al hacer merge a develop y luego a main, **eso aun no cambia la version que esta siendo desplegada desde el servidor central**

Necesitamos hacer: 

            git pull

para desplegar la version mas actualizada del main

## **Flujo completo en tu PC**

En el repositorio del servidor:

```bash
git switch develop
git pull origin develop
```
Haces los cambios, pruebas y confirmas:

```bash
git add .
git commit -m "Describir el cambio realizado"
git push origin develop
```

Luego integras a main:

```bash
git switch main
git pull origin main
git merge develop
git push origin main
```

## **Flujo en Ubuntu Server**

Debes entrar en:

            /home/admin/nomre_proyecto

Desde cualquier carpeta:

            cd ~/nombre_proyecto

Comprueba:
```bash
pwd
git status
git branch --show-current
```
Deberías ver:

            /home/admin/nombre_proyecto
            
            main

Y el repositorio debería estar limpio.



### **1. Descargar la última versión de main**
```bash
git pull --ff-only origin main
```
Uso --ff-only para impedir que el servidor cree merges accidentales. El servidor solo debe recibir la versión estable que ya preparaste en GitHub.

### **2. Reconstruir y actualizar los contenedores**
```bash
docker compose up -d --build
```
Este comando:

- Reconstruye la imagen del proyecto si cambió el código o las dependencias;
- Mantiene PostgreSQL y su volumen;
- Reemplaza el contenedor de la aplicación cuando sea necesario;
- No afecta los proyectos de otros contenedores que esten desplegados en el mismo servidor 

**No necesitas ejecutar previamente:**

            docker compose down

De hecho, normalmente **es mejor evitarlo** porque produce una interrupción más larga y elimina temporalmente la red del proyecto.

### **3. Comprobar el resultado**

            docker compose ps

Luego:

            docker compose logs --tail=100 app

Y finalmente:

            curl -I http://127.0.0.1:puerto_usado_por_el_proyecto

También puedes comprobar el dominio público:

        curl -I https://******** (url del proyecto si es que ya tiene uno)


### **Secuencia resumida**

Cada vez que publiques una nueva versión estable:
```bash
cd ~/nombre_de_proyecto
git pull --ff-only origin main
docker compose up -d --build
docker compose ps
docker compose logs --tail=100 app
curl -I http://127.0.0.1:puerto_usado_por_el_proyecto
```

### **¿Cuándo sí detener los contenedores?**

Solo sería necesario en casos concretos:

- Cambias manualmente un volumen o una red;
- Necesitas hacer mantenimiento de PostgreSQL;
- Quieres dejar dejar el proyecto fuera de servicio deliberadamente;
- Hay un error que requiere limpiar y recrear los contenedores.

### **Para detener el proyecto sin afectar los demas contenedores:**
```bash
cd ~/nombre_de_proyecto
docker compose stop
```

### **Para volver a iniciarlo:**
```bash
docker compose start
```
### **Para eliminar los contenedores y reconstruirlos:**
```bash
docker compose down
docker compose up -d --build
```
- docker compose down no elimina la base de datos **mientras no agregues -v.**

- Nunca uses esto sin intención expresa: docker compose down -v

- Ese -v eliminaría el volumen -postgres-data y con él los datos de PostgreSQL.

### **La rutina normal de actualización debe ser simplemente:**
```bash
git pull --ff-only origin main
docker compose up -d --build
```